# NIHONGA — Phase 2: Establish the Original Baseline

**Phase registered:** `Phase 2 — Establish the Original Baseline` (see [PLAN.md](PLAN.md#phase-2--establish-the-original-baseline)).
**Experimental-order step:** `2. Then we benchmark the original Qwen`.

Before modifying the model, we measure the **unmodified Qwen-Image 2.1** under fixed, fully serialized conditions.
Every later model (pruned, healed, step-distilled, quantized) is compared against these numbers under exactly the
same conditions.

## What this notebook does

1. **Freezes the evaluation protocol**: model revision, scheduler configuration, number of denoising steps, guidance,
   resolution per aspect ratio, dtype and per-prompt seeds. Serialized once, reused by every later phase.
2. **Builds the fixed internal evaluation set** from **all 1,007 prompts** of the Phase 1 pilot `internal_test` split
   (held out from training; same language mix as the pilot, ~88% EN / 12% ZH), with a fixed
   `prompt, seed, resolution, aspect_ratio, scheduler` per row.
3. **Generates the baseline images** with the original model on Modal, for:
   * the 1,000 **Qwen-Image-Bench** prompts in **both languages** (1,000 `prompt_cn` + 1,000 `prompt_en`), evaluation
     only. Chinese is the protocol of the official leaderboard (directly comparable with the public scores); English
     on the same prompts gives a paired CN-vs-EN comparison, so a language-specific degradation after compression is
     visible;
   * the internal evaluation set (1,007 prompts).

   Total: **3,007 images** per evaluated model.

   **Noise floor.** Generation is *not* bit-exact across GPU containers (same seed and conditions give slightly
   different fine details), so a fixed 1-in-10 subset (301 images) is regenerated in fresh containers and judged
   again: the resulting score difference is the **noise floor**, below which a difference between two models is not
   meaningful.
4. **Scores the images with Q-Judger**, the official Qwen-Image-Bench judge, which returns the full 56-facet score
   vector per image. From it we derive the metrics required by the plan:

   | Plan metric | Source |
   |---|---|
   | Qwen-Image-Bench overall score | Q-Judger total (per image: mean of its judged pillars; then mean over images) |
   | category-level scores | L1 pillars (Quality, Aesthetics, Alignment, Real-world Fidelity, Creative Generation) + L2 / L3 facets |
   | prompt adherence | Alignment pillar |
   | text rendering | Creative Generation / Text Rendering (L2) |
   | spatial reasoning | Alignment / Layout (L2: 2D Space, 3D Space) |
   | visual quality | Quality pillar |
   | aesthetic quality | Aesthetics pillar |

5. **Measures efficiency** on a fixed GPU: inference latency (per step and end-to-end), peak VRAM, throughput,
   number of denoising steps and model size (parameters and bytes per component).
6. **Serializes everything**: protocol, evaluation set, every generated image, raw judge outputs, parsed scores,
   efficiency measurements and a manifest with hashes.

## Outputs (`data/phase2/`)

| File | Content |
|---|---|
| `protocol.json` | frozen evaluation conditions (model revision, scheduler, steps, guidance, resolutions, dtype, GPU) |
| `internal_eval_set.jsonl` | fixed internal evaluation set (`prompt, seed, resolution, aspect_ratio, scheduler`) |
| `bench_eval_set.jsonl` | fixed Qwen-Image-Bench evaluation set (1,000 CN + 1,000 EN rows, paired seeds) |
| `images/{bench_cn,bench_en,internal}/` | every baseline image, named by prompt id |
| `smoke/`, `generator_info.json` | smoke-test images/timings; scheduler config, GPU, library versions, component sizes |
| `generation_log.jsonl` | one line per generated image: seconds, size, SHA-256 (append-only, resumable) |
| `images_rerun/`, `generation_rerun_log.jsonl` | noise-floor subset regenerated in fresh containers |
| `noise_floor_pixels.json` | pixel-level run-to-run differences of the noise-floor subset |
| `judge_calibration/` | released Qwen-Image images, our raw judgments of them, `calibration.json` (agreement report) |
| `judge_raw.jsonl` | raw Q-Judger outputs (append-only, resumable) |
| `scores.json` | parsed scores: overall, L1, L2, L3, plus the plan metrics above |
| `efficiency.json` | latency, peak VRAM, throughput, steps, model size |
| `manifest.json` | file hashes, row counts, environment, git commit |


## 1. Configuration — the frozen evaluation protocol

Defines every condition that must stay **identical** for all the models we compare, and serializes it to
`protocol.json`:

* **Model under test**: `Qwen/Qwen-Image-2.1` pinned to a commit, BF16.
* **Sampling**: the pipeline defaults of Qwen-Image 2.1 — **40 denoising steps**, **no classifier-free guidance**
  (`true_cfg_scale = 1.0`, the model is meant to be sampled without it), default flow-matching scheduler
  (its full config is recorded when the pipeline is loaded).
* **Resolution**: constant pixel area of 1024² for every aspect ratio, computed with the pipeline's own
  `calculate_dimensions` (rounded to multiples of 32), so the resolution table is exactly what the pipeline would use.
  Qwen-Image-Bench gives no aspect ratio → square 1024×1024.
* **Seeds**: internal prompts reuse the per-prompt `seed` fixed in Phase 1; each Qwen-Image-Bench prompt gets a seed
  derived from its `ID`, **shared by its CN and EN versions**, so the CN-vs-EN comparison is paired (same prompt,
  same initial noise). The initial noise is drawn on the **CPU**, so a seed gives the same noise on any GPU type.
* **Software**: the exact library versions of the local environment (diffusers pinned to a git commit) are pinned in
  the Modal image, since a library change can change the images.
* **Judge**: Q-Judger (`Qwen/Qwen-Image-Bench`) pinned to a commit, with the **official inference parameters**
  (greedy, repetition penalty 1.05, thinking enabled, ≤ 4,096 new tokens, seed 42) and the official checklists /
  prompt / score parsing from the toolkit repo pinned to a commit. Two documented deviations: it is served with
  **vLLM** instead of ms-swift (≈5–10× cheaper; calibrated against the released judge outputs in §8), and images
  larger than 1024 px are downscaled **keeping their aspect ratio** (the official code squashes them to 1024×1024;
  identical for the 1024×1024 bench images). Internal prompts, which have no official facet list, are judged on
  Quality + Aesthetics + Alignment (always judged in the bench) plus Real-world Fidelity / Creative Generation for
  our `real_world` / `creative` prompts.
* **Efficiency measurement**: GPU type, warm-up and timed runs.


In [ ]:
# Phase 2 — configuration: the frozen evaluation protocol shared by every model we will compare
import json, hashlib, time, platform, sys                          # stdlib utilities
from collections import Counter, defaultdict                        # counters for statistics
from datetime import datetime, timezone                             # timestamps for the manifest
from pathlib import Path                                            # filesystem paths
from diffusers.pipelines.qwenimage21.pipeline_qwenimage21 import calculate_dimensions   # same rounding as the pipeline

PHASE = "Phase 2 — Establish the Original Baseline"   # phase registered in every serialized artifact
PROTOCOL_VERSION = "phase2-v1"                       # bump only if a frozen condition changes (breaks comparability)
FORCE_RECOMPUTE = False                              # True -> regenerate images / re-judge even if already on disk

# Model under test: the original, unmodified Qwen-Image 2.1
MODEL_REPO = "Qwen/Qwen-Image-2.1"                                   # Hugging Face model repo
MODEL_REVISION = "790c92633540aa0cb11d9abf19eb46d861714758"          # pinned commit -> reproducible weights
DTYPE = "bfloat16"                                                   # inference dtype
NUM_INFERENCE_STEPS = 40                                             # pipeline default for Qwen-Image 2.1
TRUE_CFG_SCALE = 1.0                                                 # no CFG: 2.1 is sampled without guidance
NOISE_DEVICE = "cpu"                                                 # initial noise on CPU -> same noise on any GPU

# Resolution: constant pixel area, one (width, height) per aspect ratio, exactly as the pipeline rounds it
TARGET_AREA = 1024 * 1024                                            # = pipeline default output_resolution ** 2
ASPECT_RATIOS = ["1:1", "16:9", "9:16", "4:3", "3:4", "3:2", "2:3", "21:9"]   # every ratio used by the Phase 1 pilot
def ar_value(ar):
    """'16:9' -> 16/9."""
    w, h = ar.split(":")
    return int(w) / int(h)
RESOLUTIONS = {ar: list(calculate_dimensions(TARGET_AREA, ar_value(ar))[:2]) for ar in ASPECT_RATIOS}  # ar -> [w, h]
BENCH_ASPECT_RATIO = "1:1"                                           # Qwen-Image-Bench has no aspect ratio -> square

# Evaluation sets
BENCH_LANGS = ["cn", "en"]                                           # every bench prompt in both languages (paired)
BENCH_SEED_SALT = "nihonga-bench-v1"                                 # seed = hash(salt, ID): same seed for CN and EN
NOISE_FLOOR_EVERY = 10                                               # 1 row in 10 of each set is regenerated (noise floor)
def bench_seed(bench_id):
    """Deterministic 32-bit seed for a Qwen-Image-Bench prompt, shared by its CN and EN versions."""
    return int(hashlib.sha256(f"{BENCH_SEED_SALT}-{bench_id}".encode()).hexdigest()[:8], 16)

# Judge: Q-Judger, the official Qwen-Image-Bench judge model
JUDGE_REPO = "Qwen/Qwen-Image-Bench"                                 # Hugging Face model repo (Qwen3.6-27B based)
JUDGE_REVISION = "1b77ff83564ac4e4e8140769eacff7ff73f35f3c"          # pinned commit
JUDGE_CODE_REPO = "QwenLM/Qwen-Image-Bench"                          # official toolkit: checklists, prompt, parsing
JUDGE_CODE_REVISION = "8ab1fb47df2fba7b0cb046770a87f6323b98ecfc"     # pinned commit
JUDGE_DIMENSIONS = ["Quality", "Aesthetics", "Alignment",            # the 5 L1 pillars (official names)
                    "Real-world Fidelity", "Creative Generation"]
JUDGE_SAMPLING = {"temperature": 0.0, "top_k": 1, "top_p": 1.0,      # = official inference parameters (README)
                  "repetition_penalty": 1.05, "max_tokens": 4096, "seed": 42}
JUDGE_ENABLE_THINKING = True                                         # official: thinking mode on
JUDGE_ENGINE = "vllm==0.30.0"                                        # deviation: official uses ms-swift (validated in §8)
JUDGE_MAX_SIDE = 1024                                                # larger images downscaled, aspect ratio kept
JUDGE_MAX_MODEL_LEN = 8192                                           # ~1k image tokens + ~0.9k text + 4,096 output
JUDGE_MAX_NUM_SEQS = 64                                              # concurrent sequences (also caps CUDA graphs): the
JUDGE_MAX_BATCHED_TOKENS = 8192                                      # hybrid model's per-sequence state OOMs at vLLM defaults
INTERNAL_PILLARS = {                                                 # our dimension -> judged pillars (no official dims)
    "alignment": ["Quality", "Aesthetics", "Alignment"],
    "visual_quality": ["Quality", "Aesthetics", "Alignment"],
    "aesthetics": ["Quality", "Aesthetics", "Alignment"],
    "real_world": ["Quality", "Aesthetics", "Alignment", "Real-world Fidelity"],
    "creative": ["Quality", "Aesthetics", "Alignment", "Creative Generation"],
}

# Compute (Modal)
GEN_GPU = "H100"                                                     # generation + efficiency measurements
JUDGE_GPU = "H100"                                                   # 27B judge in BF16 (51 GiB) fits in 80 GB
JUDGE_MAX_CONTAINERS = 2                                             # parallel judge containers (~$0.6 fixed each)
JUDGE_BATCH_SIZE = 64                                                # judge requests per remote call (vLLM batches them)
EFF_WARMUP_RUNS = 3                                                  # untimed runs (compilation, caches)
EFF_TIMED_RUNS = 10                                                  # timed runs per aspect ratio
GEN_MAX_CONTAINERS = 8                                               # parallel generation containers
GEN_BATCH_SIZE = 16                                                  # rows per remote call (resumability granularity)
GEN_PACKAGES = ["torch==2.14.0", "torchvision==0.29.0",             # = local environment (uv.lock)
                "transformers==5.17.0", "accelerate==1.15.0",       # torchvision: needed by the text-encoder processor
                "diffusers @ git+https://github.com/huggingface/diffusers@bdc2bea37a36038c44452811610489ea30ede229",
                "pillow"]

# Paths
PHASE1_DIR = Path("data/phase1")                                     # Phase 1 artifacts (inputs)
INTERNAL_TEST_PATH = PHASE1_DIR / "pilot_20k" / "internal_test.jsonl"   # 1,007 held-out pilot prompts
BENCH_PROMPTS_PATH = Path("data/qwen_image_bench/prompts.jsonl")     # 1,000 bilingual bench prompts (from Phase 1)
DATA_DIR = Path("data/phase2")                                       # root of all Phase 2 artifacts
IMAGES_DIR = DATA_DIR / "images"                                     # images/{bench_cn,bench_en,internal}/<id>.png
DATA_DIR.mkdir(parents=True, exist_ok=True)
for sub in ["bench_cn", "bench_en", "internal"]:
    (IMAGES_DIR / sub).mkdir(parents=True, exist_ok=True)

# Serialize the protocol (nothing can be lost). Explicit key list, as in Phase 1
PROTOCOL_KEYS = ["PHASE", "PROTOCOL_VERSION", "FORCE_RECOMPUTE", "MODEL_REPO", "MODEL_REVISION", "DTYPE",
                 "NUM_INFERENCE_STEPS", "TRUE_CFG_SCALE", "NOISE_DEVICE", "TARGET_AREA", "ASPECT_RATIOS", "RESOLUTIONS",
                 "BENCH_ASPECT_RATIO", "BENCH_LANGS", "BENCH_SEED_SALT", "NOISE_FLOOR_EVERY", "JUDGE_REPO", "JUDGE_REVISION",
                 "JUDGE_CODE_REPO", "JUDGE_CODE_REVISION", "JUDGE_DIMENSIONS", "JUDGE_SAMPLING",
                 "JUDGE_ENABLE_THINKING", "JUDGE_ENGINE", "JUDGE_MAX_SIDE", "JUDGE_MAX_MODEL_LEN", "JUDGE_MAX_NUM_SEQS",
                 "JUDGE_MAX_BATCHED_TOKENS", "INTERNAL_PILLARS",
                 "GEN_GPU", "JUDGE_GPU", "JUDGE_MAX_CONTAINERS", "JUDGE_BATCH_SIZE", "EFF_WARMUP_RUNS", "EFF_TIMED_RUNS",
                 "GEN_MAX_CONTAINERS", "GEN_BATCH_SIZE", "GEN_PACKAGES",
                 "PHASE1_DIR", "INTERNAL_TEST_PATH", "BENCH_PROMPTS_PATH", "DATA_DIR", "IMAGES_DIR"]
PROTOCOL = {k: globals()[k] for k in PROTOCOL_KEYS}
PROTOCOL = {k: (v.as_posix() if isinstance(v, Path) else v) for k, v in PROTOCOL.items()}
(DATA_DIR / "protocol.json").write_text(json.dumps(PROTOCOL, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"{PHASE}\nProtocol saved to {DATA_DIR / 'protocol.json'}")
print(f"{NUM_INFERENCE_STEPS} steps, true_cfg_scale={TRUE_CFG_SCALE}, {DTYPE}")
for ar, (w, h) in RESOLUTIONS.items():
    print(f"  {ar:>5} -> {w}x{h}  ({w * h / 1e6:.2f} MP)")


## 2. Build and freeze the evaluation sets

Turns the two sources into **one row per image to generate**, carrying every condition needed to reproduce it
(`eval_id, prompt, language, seed, aspect_ratio, width, height, num_inference_steps, true_cfg_scale`) plus the
metadata used later to break scores down (dimension, sub-dimension, difficulty, bench facets, ...):

* **Internal set** — all 1,007 rows of the Phase 1 `internal_test` split, with their Phase 1 `seed` and `aspect_ratio`.
* **Qwen-Image-Bench** — each of the 1,000 prompts twice (`bench_cn` / `bench_en`), same seed for both versions,
  square 1024×1024. Before using the prompts we check their provenance (`source.json`: pinned revision, 1,000 prompts).

**Freezing.** The sets are fully deterministic. On re-runs they are rebuilt and compared with the files on disk: if
anything differs (e.g. Phase 1 was regenerated) the cell stops instead of silently changing the evaluation
conditions. `FORCE_RECOMPUTE = True` overwrites them on purpose.


In [ ]:
# Build the internal + Qwen-Image-Bench evaluation sets and freeze them on disk
def read_jsonl(path):
    """Read a JSONL file into a list of dicts."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def jsonl_text(rows):
    """Canonical JSONL serialization (used both to write and to compare)."""
    return "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

def gen_conditions(aspect_ratio):
    """Frozen generation conditions for one image."""
    w, h = RESOLUTIONS[aspect_ratio]
    return {"aspect_ratio": aspect_ratio, "width": w, "height": h,
            "num_inference_steps": NUM_INFERENCE_STEPS, "true_cfg_scale": TRUE_CFG_SCALE}

# 1) Internal set: every held-out pilot prompt, with the seed / aspect ratio fixed in Phase 1
internal_src = read_jsonl(INTERNAL_TEST_PATH)
assert all(r["split"] == "internal_test" for r in internal_src), "unexpected split in internal_test.jsonl"
INTERNAL_META = ["dimension", "subdimension", "facet", "difficulty", "style", "prompt_format", "hard", "keywords"]
internal_set = [{"eval_id": f"internal-{r['id']}", "set": "internal", "source_id": r["id"], "prompt": r["prompt"],
                 "language": r["language"], "seed": r["seed"], **gen_conditions(r["aspect_ratio"]),
                 **{k: r[k] for k in INTERNAL_META}}
                for r in sorted(internal_src, key=lambda r: r["id"])]

# 2) Qwen-Image-Bench: check provenance, then one row per (prompt, language) with a shared seed
bench_source = json.loads((BENCH_PROMPTS_PATH.parent / "source.json").read_text(encoding="utf-8"))
bench_src = read_jsonl(BENCH_PROMPTS_PATH)
assert bench_source["usage"] == "EVALUATION ONLY" and bench_source["n_prompts"] == len(bench_src) == 1000
assert len({r["ID"] for r in bench_src}) == 1000, "duplicate bench IDs"
LANG_FIELDS = {"cn": ("prompt_cn", "dims_cn", "zh"), "en": ("prompt_en", "dims_en", "en")}
bench_set = []
for lang in BENCH_LANGS:
    prompt_key, dims_key, language = LANG_FIELDS[lang]
    for r in sorted(bench_src, key=lambda r: r["ID"]):
        bench_set.append({"eval_id": f"bench_{lang}-{r['ID']:04d}", "set": f"bench_{lang}", "source_id": r["ID"],
                          "prompt": r[prompt_key], "language": language, "seed": bench_seed(r["ID"]),
                          **gen_conditions(BENCH_ASPECT_RATIO), "bench_dims": r[dims_key],
                          "bench_dims_en": r["dims_en"]})     # English facet names for both languages (for grouping)

# 3) Freeze: write once; afterwards any difference with the files on disk stops the notebook
EVAL_SETS = {"internal_eval_set.jsonl": internal_set, "bench_eval_set.jsonl": bench_set}
for name, rows in EVAL_SETS.items():
    path, text = DATA_DIR / name, jsonl_text(rows)
    if path.exists() and not FORCE_RECOMPUTE:
        if path.read_text(encoding="utf-8") != text:
            raise RuntimeError(f"{path} differs from the rebuilt set: the frozen evaluation conditions changed. "
                               "Investigate (was Phase 1 regenerated?) or set FORCE_RECOMPUTE=True on purpose.")
        print(f"{name}: unchanged, frozen version on disk verified")
    else:
        path.write_text(text, encoding="utf-8")
        print(f"{name}: written")

all_rows = internal_set + bench_set
assert len({r["eval_id"] for r in all_rows}) == len(all_rows), "eval_id collision"
assert all(r["seed"] == bench_seed(r["source_id"]) for r in bench_set)   # CN and EN share the seed
print(f"\n{len(all_rows):,} images per evaluated model:", dict(Counter(r["set"] for r in all_rows)))
print("internal languages:", dict(Counter(r["language"] for r in internal_set)))
print("internal aspect ratios:", dict(Counter(r["aspect_ratio"] for r in internal_set).most_common()))
print("internal dimensions:", dict(Counter(r["dimension"] for r in internal_set).most_common()))
print("\nExamples:")
for r in [internal_set[0], bench_set[0], bench_set[1000]]:
    print(" ", {k: (v[:90] + "…" if isinstance(v, str) and len(v) > 90 else v) for k, v in r.items()
                if k in ("eval_id", "prompt", "language", "seed", "width", "height")})


## 3. Modal generator — the original Qwen-Image 2.1

Defines (does not run yet) a Modal class that serves the **unmodified** model under the frozen protocol:

* **Image**: Python 3.13 (must match the local kernel because the class is shipped from the notebook with
  `serialized=True`) and the pinned `GEN_PACKAGES`. Weights go to the persistent `nihonga-hf-cache` volume
  (shared with Phase 1), so they are downloaded only once.
* **`load`** (once per container): `QwenImage21Pipeline` at the pinned revision, BF16, on the GPU.
* **`info()`**: what we must record but cannot know locally — the scheduler class and its full config, GPU name,
  library versions, and the **model size** (parameters and bytes of every component).
* **`generate(rows)`**: for each row, builds a CPU generator seeded with the row's `seed` and calls the pipeline
  with the row's `width, height, num_inference_steps, true_cfg_scale`. Returns lossless PNG bytes and the wall-clock
  time of the call (GPU-synchronized); images are saved locally by the caller.

`GEN_MAX_CONTAINERS` H100s work in parallel on batches of `GEN_BATCH_SIZE` rows.


In [ ]:
# Modal generator for the original Qwen-Image 2.1 (nothing runs remotely until a later cell)
import io
import modal

app = modal.App("nihonga-phase2-baseline")                                     # Modal app name
gen_image = (modal.Image.debian_slim(python_version="3.13")                    # = local Python (serialized=True)
             .apt_install("git")                                               # diffusers comes from a git commit
             .pip_install(*GEN_PACKAGES))                                      # pinned = local environment
hf_cache = modal.Volume.from_name("nihonga-hf-cache", create_if_missing=True)  # persistent HF weights cache
_GEN = {k: PROTOCOL[k] for k in ("MODEL_REPO", "MODEL_REVISION", "DTYPE", "NOISE_DEVICE")}   # captured by value

@app.cls(image=gen_image, gpu=GEN_GPU, volumes={"/root/.cache/huggingface": hf_cache}, timeout=3600,
         max_containers=GEN_MAX_CONTAINERS, scaledown_window=300, serialized=True)
class QwenImageGenerator:
    @modal.enter()
    def load(self):
        """Load the pinned pipeline once per container."""
        import torch
        from diffusers import QwenImage21Pipeline
        self.pipe = QwenImage21Pipeline.from_pretrained(
            _GEN["MODEL_REPO"], revision=_GEN["MODEL_REVISION"], dtype=getattr(torch, _GEN["DTYPE"])).to("cuda")
        self.pipe.set_progress_bar_config(disable=True)

    @modal.method()
    def info(self):
        """Scheduler config, environment and per-component model size."""
        import torch, diffusers, transformers
        components = {}
        for name, comp in self.pipe.components.items():
            if isinstance(comp, torch.nn.Module):
                tensors = list(comp.parameters()) + list(comp.buffers())
                components[name] = {"class": type(comp).__name__,
                                    "params": sum(p.numel() for p in comp.parameters()),
                                    "bytes": sum(t.numel() * t.element_size() for t in tensors)}
        return {"scheduler_class": type(self.pipe.scheduler).__name__,
                "scheduler_config": json.loads(json.dumps(dict(self.pipe.scheduler.config), default=str)),
                "components": components, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
                "cuda": torch.version.cuda, "diffusers": diffusers.__version__, "transformers": transformers.__version__}

    @modal.method()
    def generate(self, rows):
        """Generate one lossless PNG per row under its frozen conditions."""
        import time, torch
        results = []
        for r in rows:
            generator = torch.Generator(_GEN["NOISE_DEVICE"]).manual_seed(r["seed"])   # same noise on any GPU
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            image = self.pipe(prompt=r["prompt"], width=r["width"], height=r["height"],
                              num_inference_steps=r["num_inference_steps"], true_cfg_scale=r["true_cfg_scale"],
                              generator=generator).images[0]
            torch.cuda.synchronize()
            seconds = time.perf_counter() - t0
            buf = io.BytesIO()
            image.save(buf, format="PNG")
            results.append({"eval_id": r["eval_id"], "png": buf.getvalue(), "seconds": seconds, "size": list(image.size)})
        return results

print(f"Modal generator ready: {MODEL_REPO}@{MODEL_REVISION[:7]} on {GEN_GPU} x{GEN_MAX_CONTAINERS}, "
      f"batches of {GEN_BATCH_SIZE}")


## 4. Smoke test on Modal

Before launching 3,007 generations we check, on a handful of images, that everything behaves as the protocol says:

* **Rows**: one internal prompt per aspect ratio (8 resolutions) + the CN/EN pair of Qwen-Image-Bench prompt `ID=1`
  + the CN prompt **again** (`-repeat`).
* **Checks**: the image build and weight loading work; every image has exactly the frozen `width × height`; the
  repeated generation is **bit-identical** (same seed → same PNG) **within the same container**. (Across containers
  it is not bit-exact — see §5 and §6.)
* **Records**: `info()` (scheduler config, GPU, library versions, per-component size) → `generator_info.json`;
  images + timings + hashes → `smoke/`.
* **Estimate**: median seconds per image (first call excluded: it pays CUDA warm-up) → GPU-hours and wall-clock
  time of the full run with `GEN_MAX_CONTAINERS` containers.

**Cache:** if `smoke/smoke_results.json` and `generator_info.json` already exist, they are loaded and displayed
instead of calling Modal again (`FORCE_RECOMPUTE = True` re-runs the test).

Calls use Modal's async API (`async with app.run()`, `.remote.aio`), required inside Jupyter's running event loop.


In [ ]:
# Smoke test: a few generations to validate build, resolutions and reproducibility, and to estimate the full run
from PIL import Image
from IPython.display import display

SMOKE_DIR = DATA_DIR / "smoke"                                       # smoke-test artifacts (kept, never overwritten silently)
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

first_per_ar = {}
for r in internal_set:                                               # first internal row of every aspect ratio
    first_per_ar.setdefault(r["aspect_ratio"], r)
bench_pair = [r for r in bench_set if r["source_id"] == 1]           # bench_cn-0001 + bench_en-0001
repeat = {**bench_pair[0], "eval_id": bench_pair[0]["eval_id"] + "-repeat"}   # same conditions, generated again
smoke_rows = [first_per_ar[ar] for ar in ASPECT_RATIOS] + bench_pair + [repeat]

SMOKE_CACHED = ((SMOKE_DIR / "smoke_results.json").exists() and (DATA_DIR / "generator_info.json").exists()
                and not FORCE_RECOMPUTE)
if SMOKE_CACHED:                                                     # already run: load instead of calling Modal
    gen_info = json.loads((DATA_DIR / "generator_info.json").read_text(encoding="utf-8"))
    smoke_log = json.loads((SMOKE_DIR / "smoke_results.json").read_text(encoding="utf-8"))
    wall = None
    print(f"Smoke test already run -> loaded from {SMOKE_DIR} (recorded {gen_info['recorded_utc']})\n")
else:
    t_start = time.time()
    with modal.enable_output():
        async with app.run():
            generator = QwenImageGenerator()
            gen_info = await generator.info.remote.aio()
            smoke_results = await generator.generate.remote.aio(smoke_rows)
    wall = time.time() - t_start

    # Serialize: generator info, every image, timings and hashes
    gen_info["recorded_utc"] = datetime.now(timezone.utc).isoformat()
    (DATA_DIR / "generator_info.json").write_text(json.dumps(gen_info, indent=2, ensure_ascii=False), encoding="utf-8")
    rows_by_id = {r["eval_id"]: r for r in smoke_rows}
    smoke_log = []
    for res in smoke_results:
        (SMOKE_DIR / f"{res['eval_id']}.png").write_bytes(res["png"])
        r = rows_by_id[res["eval_id"]]
        smoke_log.append({"eval_id": res["eval_id"], "seconds": round(res["seconds"], 3), "size": res["size"],
                          "expected_size": [r["width"], r["height"]], "sha256": hashlib.sha256(res["png"]).hexdigest()})
    (SMOKE_DIR / "smoke_results.json").write_text(json.dumps(smoke_log, indent=2), encoding="utf-8")

# Checks
log = {x["eval_id"]: x for x in smoke_log}
assert all(x["size"] == x["expected_size"] for x in smoke_log), "an image does not have its frozen resolution"
reproducible = log[repeat["eval_id"]]["sha256"] == log[bench_pair[0]["eval_id"]]["sha256"]
print(f"GPU: {gen_info['gpu']} | torch {gen_info['torch']} (CUDA {gen_info['cuda']}) | "
      f"diffusers {gen_info['diffusers']} | transformers {gen_info['transformers']}")
print(f"Scheduler: {gen_info['scheduler_class']}")
for name, c in gen_info["components"].items():
    print(f"  {name:<14} {c['class']:<36} {c['params'] / 1e9:6.2f} B params  {c['bytes'] / 2**30:6.2f} GiB")
print(f"\nResolutions: all {len(smoke_log)} images match the protocol")
print(f"Reproducibility (same seed twice -> identical PNG): {reproducible}")

# Timing and estimate of the full run
secs = [x["seconds"] for x in smoke_log]
steady = sorted(secs[1:])
median_s = steady[len(steady) // 2]
n_total = len(internal_set) + len(bench_set)
gpu_hours = n_total * median_s / 3600
print(f"\nSeconds per image: first {secs[0]:.1f}, median afterwards {median_s:.1f} "
      f"({median_s / NUM_INFERENCE_STEPS:.2f} s/step)"
      + (f"; smoke wall-clock {wall / 60:.1f} min (incl. build/load)" if wall else ""))
print(f"Full run estimate: {n_total:,} images -> {gpu_hours:.1f} {GEN_GPU}-hours, "
      f"~{gpu_hours / GEN_MAX_CONTAINERS:.1f} h wall-clock with {GEN_MAX_CONTAINERS} containers")

# Contact sheet of the smoke images
thumbs = [Image.open(SMOKE_DIR / f"{x['eval_id']}.png") for x in smoke_log]
thumbs = [t.resize((int(t.width * 256 / t.height), 256)) for t in thumbs]
sheet_rows = [thumbs[:6], thumbs[6:]]                               # two rows of thumbnails
sheet = Image.new("RGB", (max(sum(t.width for t in row) for row in sheet_rows), 256 * len(sheet_rows)), "white")
for row_idx, row in enumerate(sheet_rows):
    left = 0
    for t in row:
        sheet.paste(t, (left, 256 * row_idx))
        left += t.width
display(sheet)
print(" | ".join(x["eval_id"] for x in smoke_log))


## 5. Generate the baseline images (full run, resumable)

Generates the **3,007 baseline images** (1,007 internal + 1,000 bench CN + 1,000 bench EN) with the generator of §3
under the frozen conditions of §2.

* **Resumable**: every image is written atomically (temp file + rename) to `images/<set>/<eval_id>.png` as soon as
  its batch returns, and a line is appended to `generation_log.jsonl` (seconds, size, SHA-256, timestamp).
  Re-running the cell only generates the rows that are not in the log yet **or whose file on disk no longer matches
  its logged hash** (damaged file), so an interruption loses at most the batches in flight.
  `FORCE_RECOMPUTE = True` regenerates everything.
* **Parallel**: batches of `GEN_BATCH_SIZE` rows are spread over `GEN_MAX_CONTAINERS` H100s (`.map.aio`), results
  are consumed in completion order.
* **Checks at the end**: every row has its PNG, with its frozen resolution and the hash recorded in the log.
* **Cross-container reproducibility** (measured, not asserted): the 10 smoke-test prompts were generated earlier in
  another container under the same conditions. They are **not bit-exact** across containers: GPU kernels (matmul /
  attention) can pick different reduction orders, and the tiny numerical differences accumulate over 40 denoising
  steps → same composition, different fine details. We report bit-identical count and mean |pixel difference|;
  the effect on the *scores* is quantified in §6 (noise floor).

The generation and verification logic is written as two functions (`generate_rows`, `verify_rows`) so that §6
reuses exactly the same code.


In [ ]:
# Full baseline generation: 3,007 images, streamed to disk, resumable through generation_log.jsonl
import numpy as np

GEN_LOG_PATH = DATA_DIR / "generation_log.jsonl"
eval_rows = internal_set + bench_set                                 # every image to generate (frozen conditions)
rows_by_id = {r["eval_id"]: r for r in eval_rows}

def image_path(r, root=IMAGES_DIR):
    """<root>/<set>/<eval_id>.png"""
    return root / r["set"] / f"{r['eval_id']}.png"

def file_sha256(path):
    """SHA-256 of a file on disk."""
    return hashlib.sha256(path.read_bytes()).hexdigest()

async def generate_rows(rows, root, log_path):
    """Generate every row that is not logged yet or whose file is missing/damaged; stream PNGs + log lines."""
    logged = {x["eval_id"]: x["sha256"] for x in read_jsonl(log_path)} if log_path.exists() else {}
    on_disk = {e for e in logged if image_path(rows_by_id[e], root).exists()}
    done = {e for e in on_disk if file_sha256(image_path(rows_by_id[e], root)) == logged[e]}   # intact files only
    if len(on_disk) > len(done):
        print(f"{len(on_disk) - len(done)} image(s) on disk do not match their logged hash (damaged) -> regenerated")
    todo = [r for r in rows if r["eval_id"] not in done]
    print(f"{len(rows) - len(todo):,} already generated, {len(todo):,} to generate")
    if not todo:
        return
    batches = [todo[i:i + GEN_BATCH_SIZE] for i in range(0, len(todo), GEN_BATCH_SIZE)]
    t0, n_new = time.time(), 0
    with modal.enable_output():
        async with app.run():
            with open(log_path, "a", encoding="utf-8") as log_f:
                async for results in QwenImageGenerator().generate.map.aio(batches, order_outputs=False):
                    for res in results:
                        r = rows_by_id[res["eval_id"]]
                        tmp = image_path(r, root).with_suffix(".png.tmp")   # atomic write: never a half-written PNG
                        tmp.write_bytes(res["png"])
                        tmp.replace(image_path(r, root))
                        log_f.write(json.dumps({"eval_id": r["eval_id"], "set": r["set"],
                                                "seconds": round(res["seconds"], 3), "size": res["size"],
                                                "sha256": hashlib.sha256(res["png"]).hexdigest(),
                                                "ts": datetime.now(timezone.utc).isoformat()}) + "\n")
                    log_f.flush()
                    n_new += len(results)
                    elapsed = time.time() - t0
                    print(f"{n_new:,}/{len(todo):,} images ({elapsed / 60:.1f} min, "
                          f"ETA {elapsed / n_new * (len(todo) - n_new) / 60:.1f} min)")

def verify_rows(rows, root, log_path):
    """Every row has its PNG, frozen resolution and logged hash. Returns the log (last line per eval_id)."""
    log = {x["eval_id"]: x for x in read_jsonl(log_path)}
    missing = [r["eval_id"] for r in rows if r["eval_id"] not in log or not image_path(r, root).exists()]
    assert not missing, f"{len(missing)} images missing, e.g. {missing[:3]} -> re-run this cell"
    assert all(log[r["eval_id"]]["size"] == [r["width"], r["height"]] for r in rows), "wrong resolution"
    bad = [r["eval_id"] for r in rows if file_sha256(image_path(r, root)) != log[r["eval_id"]]["sha256"]]
    assert not bad, f"{len(bad)} images do not match their logged hash, e.g. {bad[:3]} -> re-run this cell"
    return log

def pixel_diff(path_a, path_b):
    """Mean / 99th-percentile absolute RGB difference (0-255) and share of pixels differing by more than 8."""
    a = np.asarray(Image.open(path_a).convert("RGB"), dtype=np.int16)
    b = np.asarray(Image.open(path_b).convert("RGB"), dtype=np.int16)
    d = np.abs(a - b)
    return {"mean_abs": float(d.mean()), "p99_abs": float(np.percentile(d, 99)),
            "frac_px_gt8": float((d.max(axis=-1) > 8).mean())}

if FORCE_RECOMPUTE and GEN_LOG_PATH.exists():
    GEN_LOG_PATH.unlink()                                            # start from scratch on purpose
await generate_rows(eval_rows, IMAGES_DIR, GEN_LOG_PATH)
gen_log = verify_rows(eval_rows, IMAGES_DIR, GEN_LOG_PATH)
print(f"All {len(eval_rows):,} images present, correct resolution, hashes verified")

# Cross-container reproducibility: the smoke images were generated in another container, same conditions
smoke_cmp = [{"eval_id": x["eval_id"], "identical": x["sha256"] == gen_log[x["eval_id"]]["sha256"],
              **pixel_diff(SMOKE_DIR / f"{x['eval_id']}.png", image_path(rows_by_id[x["eval_id"]]))}
             for x in smoke_log if x["eval_id"] in rows_by_id]      # skips the '-repeat' row
print(f"Cross-container vs smoke test: {sum(c['identical'] for c in smoke_cmp)}/{len(smoke_cmp)} bit-identical, "
      f"mean |pixel diff| {np.mean([c['mean_abs'] for c in smoke_cmp]):.2f}/255 -> not bit-exact (noise floor: §6)")

secs = [gen_log[r["eval_id"]]["seconds"] for r in eval_rows]
print(f"Per-image seconds: mean {sum(secs) / len(secs):.2f}, min {min(secs):.2f}, max {max(secs):.2f} "
      f"-> {sum(secs) / 3600:.2f} {GEN_GPU}-hours of generation")
for s in ["internal", "bench_cn", "bench_en"]:
    print(f"  {s:<9} {sum(1 for r in eval_rows if r['set'] == s):>5,} images in {IMAGES_DIR / s}")


## 6. Noise floor — step 1: regenerate a fixed subset in fresh containers

§5 showed that generation is not bit-exact across containers. To know how much this matters **for the scores**, we
regenerate a fixed subset under exactly the same frozen conditions, in new containers:

* **Subset**: 1 row in `NOISE_FLOOR_EVERY` (= 10) of each set, in the frozen order → 101 internal + 100 bench CN +
  100 bench EN = **301 images** (~10% of the cost of the full run).
* **Storage**: `images_rerun/<set>/<eval_id>.png` + `generation_rerun_log.jsonl`, with the same resumable,
  hash-verified `generate_rows` / `verify_rows` as §5 (a new `app.run()` always starts new containers).
* **Pixel level** (here): per image, bit-identity, mean and 99th-percentile |difference| and share of pixels that
  differ by more than 8/255 → `noise_floor_pixels.json`.
* **Score level** (after the judge sections): both versions are judged, and the difference of the benchmark scores
  between the two runs is the noise floor used to decide whether a later model is really better or worse.


In [ ]:
# Noise floor, step 1: regenerate 1-in-10 rows of each set in fresh containers and measure pixel differences
RERUN_DIR = DATA_DIR / "images_rerun"                                # images_rerun/<set>/<eval_id>.png
RERUN_LOG_PATH = DATA_DIR / "generation_rerun_log.jsonl"
rerun_rows = [r for s in ["internal", "bench_cn", "bench_en"]
              for i, r in enumerate(x for x in eval_rows if x["set"] == s) if i % NOISE_FLOOR_EVERY == 0]
for s in ["internal", "bench_cn", "bench_en"]:
    (RERUN_DIR / s).mkdir(parents=True, exist_ok=True)

if FORCE_RECOMPUTE and RERUN_LOG_PATH.exists():
    RERUN_LOG_PATH.unlink()
await generate_rows(rerun_rows, RERUN_DIR, RERUN_LOG_PATH)
rerun_log = verify_rows(rerun_rows, RERUN_DIR, RERUN_LOG_PATH)

# Pixel-level run-to-run differences (baseline §5 vs rerun), per image and per set
pix = [{"eval_id": r["eval_id"], "set": r["set"],
        "identical": rerun_log[r["eval_id"]]["sha256"] == gen_log[r["eval_id"]]["sha256"],
        **pixel_diff(image_path(r), image_path(r, RERUN_DIR))} for r in rerun_rows]
summary = {}
for s in ["internal", "bench_cn", "bench_en", "all"]:
    sub = [x for x in pix if s == "all" or x["set"] == s]
    summary[s] = {"n": len(sub), "bit_identical": sum(x["identical"] for x in sub),
                  **{k: round(float(np.mean([x[k] for x in sub])), 4) for k in ("mean_abs", "p99_abs", "frac_px_gt8")}}
(DATA_DIR / "noise_floor_pixels.json").write_text(
    json.dumps({"subset_every": NOISE_FLOOR_EVERY, "summary": summary, "per_image": pix}, indent=2), encoding="utf-8")

print(f"\n{'set':<9} {'n':>4} {'identical':>9} {'mean|d|':>8} {'p99|d|':>7} {'px>8/255':>9}")
for s, v in summary.items():
    print(f"{s:<9} {v['n']:>4} {v['bit_identical']:>9} {v['mean_abs']:>8.2f} {v['p99_abs']:>7.1f} "
          f"{v['frac_px_gt8'] * 100:>8.2f}%")

# The pair with the largest difference, side by side (baseline | rerun)
worst = max(pix, key=lambda x: x["mean_abs"])
a, b = Image.open(image_path(rows_by_id[worst["eval_id"]])), Image.open(image_path(rows_by_id[worst["eval_id"]], RERUN_DIR))
pair = Image.new("RGB", (a.width * 2, a.height), "white")
pair.paste(a, (0, 0))
pair.paste(b, (a.width, 0))
display(pair.resize((pair.width * 384 // pair.height, 384)))
print(f"Largest difference: {worst['eval_id']} (mean |d| {worst['mean_abs']:.2f}) — left: baseline, right: rerun")


## 7. Q-Judger — official toolkit + Modal judge definition

**Official toolkit, vendored.** The evaluation prompts and the score parsing are *not* re-implemented: the official
`checklists.py` (system prompt, user-prompt template, the 5 fixed pillar checklists = 56 facets) and `score_utils.py`
(JSON extraction after `</think>`, 0/1/2 → 0/60/100 mapping, L3 → L2 → L1 → total aggregation, fixes for misplaced
facets) are downloaded once from `QwenLM/Qwen-Image-Bench` at the pinned commit into `third_party/qwen_image_bench/`
(with `SOURCE.json`: repo, commit, SHA-256 per file) and imported from there.

**How one judgment works** (same as the official `judge.py`): one request per *(image, pillar)*; the user text is the
official template filled with the prompt, the pillar name and its checklist, and contains an `<image>` placeholder
where the image goes. The judge thinks, then answers with a JSON of facet scores (or `N/A`).

**Modal class `QJudger`** (defined here, run in §8):

* Image: Python 3.13 + `vllm==0.30.0`; `VLLM_USE_FLASHINFER_SAMPLER=0` (as in Phase 1: the FlashInfer sampler needs
  `nvcc`, absent from `debian_slim`). Weights cached in the `nihonga-hf-cache` volume.
* `load`: `Qwen/Qwen-Image-Bench` at the pinned revision, one image per prompt, `max_model_len = 8192`,
  **`max_num_seqs = 64`, `max_num_batched_tokens = 8192`**: with vLLM's defaults (CUDA graphs for up to 512
  concurrent sequences, 16,384 tokens per step) the per-sequence state of this hybrid (linear-attention) model does
  not fit next to the 51 GiB of weights and the engine runs out of memory at start-up. 64 concurrent requests is
  more than the KV cache holds for our long (thinking) requests anyway.
* `judge(requests)`: splits the user text at `<image>` and interleaves the image, applies the chat template with
  `enable_thinking=True`, decodes with the official sampling parameters, and returns the raw text, number of output
  tokens, finish reason and the inference time of the batch (a `length` finish means the 4,096-token budget was hit).


In [ ]:
# Official Qwen-Image-Bench toolkit (vendored at a pinned commit) + Modal Q-Judger served with vLLM
import importlib.util, urllib.request

QIB_DIR = Path("third_party/qwen_image_bench")                       # vendored official files
QIB_FILES = ["checklists.py", "score_utils.py", "LICENSE"]           # prompts/checklists, score parsing, Apache-2.0
QIB_DIR.mkdir(parents=True, exist_ok=True)
for name in QIB_FILES:
    if not (QIB_DIR / name).exists():                                # downloaded once, then read from disk
        url = f"https://raw.githubusercontent.com/{JUDGE_CODE_REPO}/{JUDGE_CODE_REVISION}/{name}"
        (QIB_DIR / name).write_bytes(urllib.request.urlopen(url).read())
(QIB_DIR / "SOURCE.json").write_text(json.dumps(
    {"repo": JUDGE_CODE_REPO, "revision": JUDGE_CODE_REVISION,
     "sha256": {n: hashlib.sha256((QIB_DIR / n).read_bytes()).hexdigest() for n in QIB_FILES}}, indent=2),
    encoding="utf-8")

def load_module(name, path):
    """Import a vendored file under a unique module name."""
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

qib_checklists = load_module("qib_checklists", QIB_DIR / "checklists.py")   # SYSTEM_PROMPT, USER_PROMPT_TEMPLATE, ...
qib_scores = load_module("qib_scores", QIB_DIR / "score_utils.py")          # extract/fix/compute/aggregate
assert list(qib_checklists.DIM_TO_CHECKLIST) == ["Quality", "Aesthetics", "Alignment",
                                                 "Real-world Fidelity", "Creative Generation"]
assert set(JUDGE_DIMENSIONS) == set(qib_checklists.DIM_TO_CHECKLIST)
assert qib_checklists.USER_PROMPT_TEMPLATE.count("<image>") == 1
n_facets = sum(len(v) for v in qib_scores.CHECKLIST_L3_TO_L2.values())
print(f"Official toolkit @ {JUDGE_CODE_REVISION[:7]}: {len(JUDGE_DIMENSIONS)} pillars, {n_facets} facets")

# Modal judge (nothing runs remotely until §8)
judge_image = (modal.Image.debian_slim(python_version="3.13")        # = local Python (serialized=True)
               .pip_install(JUDGE_ENGINE)
               .env({"VLLM_USE_FLASHINFER_SAMPLER": "0"}))            # FlashInfer sampler needs nvcc (absent)
_JUDGE = {"repo": JUDGE_REPO, "revision": JUDGE_REVISION, "sampling": JUDGE_SAMPLING,
          "thinking": JUDGE_ENABLE_THINKING, "max_model_len": JUDGE_MAX_MODEL_LEN,
          "max_num_seqs": JUDGE_MAX_NUM_SEQS, "max_batched_tokens": JUDGE_MAX_BATCHED_TOKENS}   # captured by value

@app.cls(image=judge_image, gpu=JUDGE_GPU, volumes={"/root/.cache/huggingface": hf_cache}, timeout=3600,
         max_containers=JUDGE_MAX_CONTAINERS, scaledown_window=300, serialized=True)
class QJudger:
    @modal.enter()
    def load(self):
        """Load Q-Judger once per container."""
        from vllm import LLM, SamplingParams
        self.llm = LLM(model=_JUDGE["repo"], revision=_JUDGE["revision"], max_model_len=_JUDGE["max_model_len"],
                       max_num_seqs=_JUDGE["max_num_seqs"], max_num_batched_tokens=_JUDGE["max_batched_tokens"],
                       limit_mm_per_prompt={"image": 1}, gpu_memory_utilization=0.90, seed=_JUDGE["sampling"]["seed"])
        self.params = SamplingParams(**_JUDGE["sampling"])

    @modal.method()
    def judge(self, requests):
        """requests: [{key, system, user_text (with one <image>), png}]
        -> [{key, output, n_out_tokens, finish_reason, batch_seconds, batch_size}]"""
        import io, time
        from PIL import Image
        conversations = []
        for q in requests:
            before, after = q["user_text"].split("<image>")          # the image goes where the template says
            image = Image.open(io.BytesIO(q["png"])).convert("RGB")
            conversations.append([
                {"role": "system", "content": q["system"]},
                {"role": "user", "content": [{"type": "text", "text": before},
                                             {"type": "image_pil", "image_pil": image},
                                             {"type": "text", "text": after}]}])
        t0 = time.perf_counter()
        outputs = self.llm.chat(conversations, self.params, use_tqdm=False,
                                chat_template_kwargs={"enable_thinking": _JUDGE["thinking"]})
        seconds = time.perf_counter() - t0                            # pure inference time of the whole batch
        return [{"key": q["key"], "output": o.outputs[0].text, "n_out_tokens": len(o.outputs[0].token_ids),
                 "finish_reason": o.outputs[0].finish_reason, "batch_seconds": round(seconds, 2),
                 "batch_size": len(requests)} for q, o in zip(requests, outputs)]

print(f"Modal judge ready: {JUDGE_REPO}@{JUDGE_REVISION[:7]} with {JUDGE_ENGINE} on {JUDGE_GPU} "
      f"x{JUDGE_MAX_CONTAINERS}, batches of {JUDGE_BATCH_SIZE}")


## 8. Judge calibration + smoke test (vs the officially released judgments)

Our judge differs from the official pipeline in one place: it is served with **vLLM** instead of ms-swift. Before
judging our 3,007 images we check that it reproduces the **official judgments**, which the benchmark dataset
releases together with the images of every leaderboard model:

* **Sample**: every 20th Qwen-Image-Bench ID (1, 21, …, 981) → **50 prompts**; the images of the leaderboard model
  **Qwen-Image** (generated from `prompt_cn`, the leaderboard protocol) are downloaded from the dataset at the pinned
  revision into `judge_calibration/images/`.
* **Requests**: exactly as the official `judge.py`: the pillars listed in each prompt's `dims_en`, the official
  system prompt / template / checklist, `prompt_cn`. ≈ 205 requests, sent in **one call to one container** (vLLM
  batches them internally; one container = the load cost is paid once).
* **Comparison** (both sides parsed with the same official `score_utils`): per pillar, mean score ours vs official
  and the per-image correlation; the per-image total; facet-level exact agreement (0 / 1 / 2 / N/A); parse failures
  and truncated outputs (4,096-token budget hit). → `judge_calibration/calibration.json`.
* **Throughput → cost of the full run**: measured inference seconds per request × the exact number of requests of
  the full run (bench from their `dims_en`, internal from `INTERNAL_PILLARS`, plus the noise-floor subset), at Modal's
  H100 price.

**Cache:** raw outputs are appended to `judge_calibration/judge_raw.jsonl`; re-running the cell only sends the
requests that are not there yet.


In [ ]:
# Judge calibration: our vLLM Q-Judger vs the officially released judgments of the leaderboard model Qwen-Image
import io, shutil
from huggingface_hub import hf_hub_download

CALIB_DIR = DATA_DIR / "judge_calibration"                           # calibration artifacts
(CALIB_DIR / "images").mkdir(parents=True, exist_ok=True)
CALIB_RAW_PATH = CALIB_DIR / "judge_raw.jsonl"                       # our raw judgments (append-only)
CALIB_MODEL = "Qwen-Image"                                           # leaderboard model with released images + judgments
CALIB_EVERY = 20                                                     # every 20th bench ID -> 50 prompts
H100_USD_PER_S = 0.001097                                            # Modal H100 price (modal.com/pricing, 2026-09-26)
RESPONSE_PREFIX = {"Quality": "quality_response_", "Aesthetics": "aesthetics_response_",   # released judgment columns
                   "Alignment": "alignment_response_", "Real-world Fidelity": "real_world_fidelity_response_",
                   "Creative Generation": "creative_generation_response_"}

def judge_png(path):
    """Image -> PNG bytes as the judge sees it: RGB, longer side <= JUDGE_MAX_SIDE with the aspect ratio kept."""
    image = Image.open(path).convert("RGB")
    if max(image.size) > JUDGE_MAX_SIDE:
        scale = JUDGE_MAX_SIDE / max(image.size)
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return buf.getvalue()

def judge_requests(key, prompt, pillars, png):
    """One request per pillar, built exactly like the official judge.py."""
    return [{"key": f"{key}|{pillar}", "system": qib_checklists.SYSTEM_PROMPT, "png": png,
             "user_text": qib_checklists.USER_PROMPT_TEMPLATE.format(
                 prompt=prompt, level1_dim=pillar, format_checklist=qib_checklists.DIM_TO_CHECKLIST[pillar])}
            for pillar in pillars]

def parse_judgment(text, pillar):
    """Official parsing: JSON after </think> -> fixed L2/L3 hierarchy -> L1/L2/L3 scores (None if unparseable)."""
    score_json = qib_scores.extract_json_from_response(text) if text else None
    if not score_json:
        return None
    return qib_scores.compute_dimension_score(qib_scores.fix_score_json(score_json, pillar))

# 1) Sample: released Qwen-Image images + released judgments for every 20th bench ID
bench_raw_path = hf_hub_download(bench_source["repo"], bench_source["file"], repo_type="dataset",
                                 revision=bench_source["revision"])
calib_src = {}
with open(bench_raw_path, encoding="utf-8") as f:                    # 185 MB: keep only the sampled rows
    for line in f:
        r = json.loads(line)
        if r["ID"] % CALIB_EVERY == 1:
            calib_src[r["ID"]] = r
calib_requests, official = [], {}
for bench_id, r in sorted(calib_src.items()):
    local = CALIB_DIR / "images" / Path(r[CALIB_MODEL]).name
    if not local.exists():
        shutil.copy(hf_hub_download(bench_source["repo"], r[CALIB_MODEL], repo_type="dataset",
                                    revision=bench_source["revision"]), local)
    pillars = list(qib_checklists.parse_dims_by_level1(r["dims_en"]))   # official: pillars listed in dims_en
    calib_requests += judge_requests(f"calib-{bench_id:04d}", r["prompt_cn"], pillars, judge_png(local))
    for pillar in pillars:
        official[f"calib-{bench_id:04d}|{pillar}"] = parse_judgment(r[RESPONSE_PREFIX[pillar] + CALIB_MODEL], pillar)
print(f"{len(calib_src)} prompts, {len(calib_requests)} judge requests "
      f"({Counter(q['key'].split('|')[1] for q in calib_requests)})")

# 2) Judge them: one call, one container (resumable through judge_raw.jsonl)
done = {x["key"] for x in read_jsonl(CALIB_RAW_PATH)} if CALIB_RAW_PATH.exists() else set()
todo = [q for q in calib_requests if q["key"] not in done]
print(f"{len(done)} already judged, {len(todo)} to judge")
if todo:
    t0 = time.time()
    with modal.enable_output():
        async with app.run():
            results = await QJudger().judge.remote.aio(todo)
    with open(CALIB_RAW_PATH, "a", encoding="utf-8") as f:
        for res in results:
            f.write(json.dumps({**res, "ts": datetime.now(timezone.utc).isoformat()}, ensure_ascii=False) + "\n")
    print(f"Judged {len(results)} requests in {(time.time() - t0) / 60:.1f} min wall-clock "
          f"(incl. container start, weight download on first run, model load)")
raw = {x["key"]: x for x in read_jsonl(CALIB_RAW_PATH)}

# 3) Compare with the official judgments (both parsed with the official score_utils)
ours = {k: parse_judgment(raw[k]["output"], k.split("|")[1]) for k in official}
n_parse_fail = sum(v is None for v in ours.values())
n_truncated = sum(raw[k]["finish_reason"] == "length" for k in official)
def l1(d):
    return None if d is None else d["level1_score"]

report = {"model": CALIB_MODEL, "n_prompts": len(calib_src), "n_requests": len(official),
          "parse_failures_ours": n_parse_fail, "parse_failures_official": sum(v is None for v in official.values()),
          "truncated_ours": n_truncated, "pillars": {}}
print(f"\nParse failures: ours {n_parse_fail}, official {report['parse_failures_official']}; "
      f"truncated (4,096 tokens): {n_truncated}")
print(f"{'pillar':<20} {'n':>3} {'ours':>7} {'official':>9} {'diff':>6} {'r':>6}")
for pillar in JUDGE_DIMENSIONS:
    pairs = [(l1(ours[k]), l1(official[k])) for k in official
             if k.endswith("|" + pillar) and l1(ours[k]) is not None and l1(official[k]) is not None]
    if not pairs:
        continue
    a, b = np.array(pairs).T
    r_ = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else float("nan")
    report["pillars"][pillar] = {"n": len(pairs), "ours": float(a.mean()), "official": float(b.mean()), "pearson": r_}
    print(f"{pillar:<20} {len(pairs):>3} {a.mean():>7.2f} {b.mean():>9.2f} {a.mean() - b.mean():>+6.2f} {r_:>6.2f}")

# Per-image total (official aggregation: mean of the image's pillar scores), then mean over images
def image_totals(scores):
    per_image = defaultdict(dict)
    for k, v in scores.items():
        if v is not None:
            per_image[k.split("|")[0]][k.split("|")[1]] = v
    return {img: qib_scores.aggregate_total_score(d) for img, d in per_image.items()}
tot_ours, tot_off = image_totals(ours), image_totals(official)
common = [i for i in tot_ours if i in tot_off and tot_ours[i] is not None and tot_off[i] is not None]
a, b = np.array([tot_ours[i] for i in common]), np.array([tot_off[i] for i in common])
report["total"] = {"n": len(common), "ours": float(a.mean()), "official": float(b.mean()),
                   "pearson": float(np.corrcoef(a, b)[0, 1])}
print(f"{'TOTAL (per image)':<20} {len(common):>3} {a.mean():>7.2f} {b.mean():>9.2f} {a.mean() - b.mean():>+6.2f} "
      f"{report['total']['pearson']:>6.2f}")

# Facet-level agreement on the raw categories 0 / 60 / 100 / N/A
facet_pairs = []
for k in official:
    if ours[k] is None or official[k] is None:
        continue
    for l2, facets in official[k]["level3_scores"].items():
        for l3, v_off in facets.items():
            v_ours = ours[k]["level3_scores"].get(l2, {}).get(l3, "missing")
            if v_ours != "missing":
                facet_pairs.append((v_ours, v_off))
exact = sum(x == y for x, y in facet_pairs) / len(facet_pairs)
scored = [(x, y) for x, y in facet_pairs if x is not None and y is not None]
report["facets"] = {"n": len(facet_pairs), "exact_agreement": exact,
                    "exact_agreement_scored": sum(x == y for x, y in scored) / len(scored),
                    "within_one_level_scored": sum(abs(x - y) <= 60 for x, y in scored) / len(scored)}
print(f"\nFacets: {len(facet_pairs)} compared, exact agreement {exact:.1%} (incl. N/A); "
      f"on facets scored by both: exact {report['facets']['exact_agreement_scored']:.1%}, "
      f"within one level {report['facets']['within_one_level_scored']:.1%}")

# 4) Throughput -> cost of the full judging run
batch_runs = {(x["batch_seconds"], x["batch_size"]) for x in raw.values()}
sec_per_request = sum(s for s, _ in batch_runs) / sum(n for _, n in batch_runs)
out_tokens = [x["n_out_tokens"] for x in raw.values()]
n_full = (sum(len(qib_checklists.parse_dims_by_level1(r["bench_dims_en"])) for r in bench_set)
          + sum(len(INTERNAL_PILLARS[r["dimension"]]) for r in internal_set))
n_rerun = (sum(len(qib_checklists.parse_dims_by_level1(r["bench_dims_en"])) for r in rerun_rows if r["set"] != "internal")
           + sum(len(INTERNAL_PILLARS[rows_by_id[r["eval_id"]]["dimension"]]) for r in rerun_rows if r["set"] == "internal"))
gpu_s = (n_full + n_rerun) * sec_per_request
report["throughput"] = {"sec_per_request": sec_per_request, "mean_out_tokens": float(np.mean(out_tokens)),
                        "full_requests": n_full, "rerun_requests": n_rerun,
                        "estimated_gpu_hours": gpu_s / 3600, "estimated_usd": gpu_s * H100_USD_PER_S}
print(f"\nThroughput: {sec_per_request:.2f} s/request (batch of {max(n for _, n in batch_runs)}), "
      f"mean {np.mean(out_tokens):.0f} output tokens")
print(f"Full run: {n_full:,} + {n_rerun:,} (noise floor) requests -> {gpu_s / 3600:.1f} H100-hours "
      f"≈ ${gpu_s * H100_USD_PER_S:.0f} of inference (+ ~$0.6 per container for start/load/idle)")
(CALIB_DIR / "calibration.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

# One example side by side: ours vs official facet scores
k = next(k for k in official if k.endswith("|Alignment") and ours[k] and official[k])
print(f"\nExample {k}:")
for l2, facets in official[k]["level3_scores"].items():
    for l3, v in facets.items():
        print(f"  {l2:<12} / {l3:<26} ours {str(ours[k]['level3_scores'].get(l2, {}).get(l3)):>6}   official {str(v):>6}")


## 9. Calibration analysis — is the vLLM judge a faithful substitute?

**How.** The differences of §8 are computed on 50 images only, so each one needs an uncertainty: per pillar and for
the per-image total we take the paired differences *ours − official* and compute a **95% bootstrap confidence
interval** of their mean (5,000 resamples of the images). A difference whose interval contains 0 is not
distinguishable from sampling noise. We also break the largest gap (Aesthetics) down by facet. Everything is added
to `judge_calibration/calibration.json` under `analysis`.

**Findings of the calibration run (2026-09-26).**

* **Total**: ours 50.53 vs official 50.42 (**+0.11**), per-image correlation 0.86 — on the headline number the two
  judges agree.
* **Pillars**: every 95% interval contains 0 (Quality +0.5, Aesthetics −2.5, Alignment +1.2, Real-world Fidelity −0.3,
  Creative Generation +2.0) → no significant bias. Aesthetics is the one to watch: all six of its facets come out
  1–4 points lower, i.e. our judge may be slightly stricter on aesthetics. This does not affect our comparisons (every
  NIHONGA model is scored by the same judge, so a constant lean cancels out); it only matters when putting our
  Aesthetics number next to a public leaderboard number.
* **Quality** has the lowest correlation (0.51) but a matching mean: quality varies little between images (sd 12.5)
  and has only 6 coarse facets, so one facet flipping Pass ↔ Excel moves an image by 10–20 points.
* **Facets**: 92% exact agreement (incl. N/A); on facets scored by both, 89% exact and **99% within one level** —
  disagreements are borderline Pass↔Excel / Fail↔Pass calls, not systematic misreadings. 0 parse failures,
  0 truncated outputs; same output length as the official judge (~550 tokens).
* **Throughput**: 0.48 s per request on one H100 → the full run (12,769 requests incl. the noise-floor subset) is
  ≈ 1.7 H100-hours (≈ $7 of inference, plus ≈ $0.6 per container) → `JUDGE_MAX_CONTAINERS` set to 2.

**Conclusion**: the vLLM-served Q-Judger reproduces the official Qwen-Image-Bench scoring and is used for all
NIHONGA evaluations. Note that greedy decoding in vLLM is not guaranteed to be identical across different batch
compositions, so the judge contributes some run-to-run variability of its own; the noise floor of §6 is measured
end-to-end (generation + judging), which is the variability that matters when comparing models.

**Implications for comparing NIHONGA models.**

* The possible lean is a property of the **judge**, not of the baseline model: the baseline is the original,
  unmodified Qwen-Image 2.1, and every later model (pruned, healed, step-distilled, quantized) is scored by **the same**
  vLLM Q-Judger with the same settings. What matters is the difference *compressed − baseline*, so a constant offset of
  the judge is present in both numbers and **cancels out**.
* The cancellation assumes the lean is roughly constant. If the judge were stricter mainly on borderline
  Pass↔Excel calls, a model with fewer excellent images would be affected slightly less, so the cancellation would not
  be perfect — but that residual is a fraction of an already non-significant ~2.5 points, far below the drops that
  matter for compression; the noise floor (§6) defines what counts as a real difference anyway.
* **The judge must stay frozen**: same judge revision, same toolkit commit, same vLLM version and the same settings
  for every evaluation (all pinned in `protocol.json`). If any of them ever has to change, the baseline must be
  re-judged with the new setup before any comparison.
* The lean only matters when quoting our absolute Aesthetics number next to the public leaderboard (it may read
  ~2.5 points low); it does not matter for the internal comparisons that drive the project.


In [ ]:
# Calibration analysis: bootstrap confidence intervals of ours - official, and the Aesthetics facets
rng = np.random.default_rng(0)                                       # fixed: the analysis is reproducible
N_BOOTSTRAP = 5000

def bootstrap_ci(diffs):
    """95% bootstrap CI of the mean of paired differences."""
    diffs = np.asarray(diffs, dtype=float)
    means = diffs[rng.integers(0, len(diffs), (N_BOOTSTRAP, len(diffs)))].mean(axis=1)
    return [float(x) for x in np.percentile(means, [2.5, 97.5])]

analysis = {"n_bootstrap": N_BOOTSTRAP, "pillars": {}, "aesthetics_facets": {}}
print(f"{'':<20} {'n':>3} {'ours-off':>9} {'95% CI':>17} {'sd(off)':>8} {'MAE':>6}")
for pillar in JUDGE_DIMENSIONS:
    keys = [k for k in official if k.endswith("|" + pillar) and l1(ours[k]) is not None and l1(official[k]) is not None]
    d = np.array([l1(ours[k]) - l1(official[k]) for k in keys])
    lo, hi = bootstrap_ci(d)
    sd_off = float(np.std([l1(official[k]) for k in keys]))
    analysis["pillars"][pillar] = {"n": len(d), "mean_diff": float(d.mean()), "ci95": [lo, hi],
                                   "sd_official": sd_off, "mae": float(np.abs(d).mean())}
    print(f"{pillar:<20} {len(d):>3} {d.mean():>+9.2f}   [{lo:+6.2f}, {hi:+6.2f}] {sd_off:>8.1f} {np.abs(d).mean():>6.1f}")
d = np.array([tot_ours[i] - tot_off[i] for i in common])
lo, hi = bootstrap_ci(d)
analysis["total"] = {"n": len(d), "mean_diff": float(d.mean()), "ci95": [lo, hi], "mae": float(np.abs(d).mean())}
print(f"{'TOTAL (per image)':<20} {len(d):>3} {d.mean():>+9.2f}   [{lo:+6.2f}, {hi:+6.2f}] {'':>8} {np.abs(d).mean():>6.1f}")

# Which facets drive the Aesthetics gap?
print("\nAesthetics facets (images where both judges gave a score):")
for l3, l2 in qib_scores.CHECKLIST_L3_TO_L2["Aesthetics"].items():
    pairs = [(ours[k]["level3_scores"].get(l2, {}).get(l3, "missing"), official[k]["level3_scores"].get(l2, {}).get(l3))
             for k in official if k.endswith("|Aesthetics") and ours[k] and official[k]]
    scored = [(a, b) for a, b in pairs if a not in (None, "missing") and b is not None]
    na_disagree = sum((a is None) != (b is None) for a, b in pairs if a != "missing")
    if not scored:
        continue
    o, f_ = np.mean([a for a, _ in scored]), np.mean([b for _, b in scored])
    exact = float(np.mean([a == b for a, b in scored]))
    analysis["aesthetics_facets"][l3] = {"n": len(scored), "ours": float(o), "official": float(f_),
                                         "exact": exact, "na_disagreements": na_disagree}
    print(f"  {l3:<24} n={len(scored):>2}  ours {o:5.1f}  official {f_:5.1f}  ({o - f_:+5.1f})  "
          f"exact {exact:.0%}  N/A disagreements {na_disagree}")

report["analysis"] = analysis
(CALIB_DIR / "calibration.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
n_sig = sum(not (v["ci95"][0] <= 0 <= v["ci95"][1]) for v in analysis["pillars"].values())
print(f"\nPillars with a significant difference (95% CI excludes 0): {n_sig}/{len(analysis['pillars'])}")
